# Submission 2 — Project AI / Algorithmic & Behavioral Model (5%)

**Course:** RBB2013 Digital Twin — May 2026
**Group project — SmartClean Twin:** Digital Twin of a mobile cleaning robot (topic 2)

**Team Members:**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |
**Repository:** https://github.com/KAI-UTP/smartclean-twin



## 1. Five AI models — three ML paradigms

| Model | Paradigm | Inputs | Output | Validation |
|---|---|---|---|---|
| Motor health classifier | Supervised classification (Random Forest) | current, temperature, speed, brush, pump, battery A | NORMAL / HIGH_LOAD / OVERHEATED / FAULT | 100% test accuracy |
| Dirt level classifier | Supervised classification | dirt score | CLEAN / MODERATE / DIRTY | 99.9% |
| Health state classifier | Supervised classification (pipeline: StandardScaler + RF) | 9 mapped sensor features | NORMAL / WARNING / CRITICAL | 90.8% accuracy, stratified 80/20 |
| RUL regressor | Supervised regression (pipeline) | same 9 features | remaining useful life (minutes) | R² = 0.91, MAE 6.6 min |
| Anomaly detector | **Unsupervised** | current, temperature, speed, battery V/A | anomaly score + flag | 100% fault detection, 0% false alarms |

**Methodology:** labelled synthetic datasets generated with documented,
physics-motivated rules (e.g. OVERHEATED = temperature > 70 °C); stratified
80/20 train/validation split; scaling inside pipelines to prevent leakage;
models trained at Docker build time — baked into the image, fully reproducible.
The anomaly detector is trained on **normal operation only** (per-sensor
mean/σ; > 4.5σ on any sensor = anomaly) — it flags faults it was never shown.

Training code: `services/ai-service/train_model.py`. Inference: `predictor.py`.


## 2. Behavioral model (algorithmic layer)

Beyond ML, the twin has behavioral logic:
- **State engine rules** — 11-dimension twin state derived from telemetry
  (safety, battery, motion, mission, cleaning, connection, twin quality...)
- **Autonomous battery behaviour** — return home < 20% SoC, dock-charge at
  10%/min, resume at 80% (simulator behavioral model)
- **Trend forecasts** — battery minutes-to-empty and cleaning
  minutes-to-finish from 60s rolling rates
- **Recommendation engine** — combines all model outputs into one operator
  advice string (STOP / inspect / return to dock / schedule maintenance / normal)


## 3. Live evidence — predictions on streaming data (1/second)

In [1]:
import json, time, urllib.request

INFLUX = "http://localhost:8086/api/v2/query?org=smartclean"
TOKEN = "smartclean-super-secret-token"

def flux_query(q):
    req = urllib.request.Request(INFLUX, data=q.encode(),
        headers={"Authorization": f"Token {TOKEN}",
                 "Content-Type": "application/vnd.flux", "Accept": "application/csv"})
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.read().decode()

def show_last(measurement, range_s=30):
    q = (f'from(bucket: "smartclean_twin") |> range(start: -{range_s}s) '
         f'|> filter(fn: (r) => r._measurement == "{measurement}") |> last()')
    n = 0
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 7 and p[1] == "_result":
            print(f"  {p[7]:28s} = {p[6]}")
            n += 1
    if n == 0:
        print("  (no data in window — is docker compose up?)")

print("Helpers loaded.")


Helpers loaded.


In [2]:
print("robot_prediction (all live AI outputs):")
show_last("robot_prediction")


robot_prediction (all live AI outputs):
  anomaly_score                = 2.8964
  dirt_level                   = MODERATE
  dirt_level_confidence        = 1
  health_state                 = NORMAL
  health_state_confidence      = 0.9518
  is_anomaly                   = 0
  minutes_to_empty             = 1083.9
  minutes_to_finish            = 1.6
  motor_health                 = NORMAL
  motor_health_confidence      = 0.95
  predicted_rul_minutes        = 115
  recommendation               = Normal operation — no action needed


## 4. What-if simulation — models answering hypothetical scenarios

The defining Digital Twin capability: query the models without touching the
robot (`POST /whatif` on the AI service).


In [3]:
def whatif(**scenario):
    req = urllib.request.Request("http://localhost:8003/whatif",
        data=json.dumps(scenario).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())["prediction"]

for label, sc in [
    ("healthy robot", dict(motor_temperature_c=40, motor_current_a=0.8, battery_soc=90)),
    ("overheating under load", dict(motor_temperature_c=90, motor_current_a=3.6, battery_soc=40)),
    ("low battery + low water", dict(battery_soc=15, water_level_pct=5)),
]:
    p = whatif(**sc)
    print(f"Scenario: {label}")
    print(f"  health={p['health_state_prediction']}  RUL={p['predicted_rul_minutes']} min  "
          f"anomaly={p['is_anomaly']}")
    print(f"  -> {p['recommendation']}")
    print()


Scenario: healthy robot
  health=NORMAL  RUL=108.8 min  anomaly=False
  -> Normal operation — no action needed



Scenario: overheating under load
  health=WARNING  RUL=30.3 min  anomaly=True
  -> Sensor anomaly detected — verify sensors and inspect robot

Scenario: low battery + low water
  health=WARNING  RUL=14.7 min  anomaly=False
  -> Schedule maintenance soon — monitor temperature and load



## 5. Model reaction to a live fault

Inject a motor overload → classifier and anomaly detector must react within
seconds on real streaming data (not a canned example).


In [4]:
def inject(fault):
    req = urllib.request.Request("http://localhost:8004/fault",
        data=json.dumps({"fault": fault}).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    urllib.request.urlopen(req, timeout=5)

inject("motor")
print("Motor fault injected, waiting 15 s ...")
time.sleep(15)
q = ('from(bucket: "smartclean_twin") |> range(start: -10s) '
     '|> filter(fn: (r) => r._measurement == "robot_prediction" and '
     '(r._field == "motor_health" or r._field == "anomaly_score" or '
     'r._field == "health_state" or r._field == "recommendation")) |> last()')
for line in flux_query(q).splitlines():
    p = line.split(",")
    if len(p) > 7 and p[1] == "_result":
        print(f"  {p[7]} = {p[6]}")
inject("clear")
print("Fault cleared.")


Motor fault injected, waiting 15 s ...


  anomaly_score = -10.8727
  health_state = WARNING
  motor_health = HIGH_LOAD
  recommendation = Sensor anomaly detected — verify sensors and inspect robot
Fault cleared.


## Rubric Mapping — Skilled (5) Level

| Skilled (5) criterion | Where demonstrated in this submission |
|---|---|
| AI / behavioral model defined and justified | Section 1 — 5 models, 3 ML paradigms, with rationale |
| Features and labels selected & justified | Section 1 methodology — documented physics-motivated labelling rules |
| Model trained with train/validation split | Stratified 80/20 split; training reproducible at Docker build |
| Validation metrics reported | Accuracies, R², MAE per model (Section 1 table) |
| Model integrated into the live twin | Section 3 — predictions on streaming data every second |
| Behavioral model beyond ML | Section 2 — state rules, autonomous charging, forecasts, recommendations |
| Model exercised on demand | Section 4 what-if scenarios; Section 5 live fault reaction |
